# 07 — Exploratory analysis: the questions behind the Democratic Funnel

**The Democratic Funnel** · Team UL · DIRISA Student Datathon Challenge 2026

**Reads:** `data/processed/master.csv` (and, if present, `data/interim/boundaries.gpkg`,
`data/interim/atlas_registration_activity_2016.csv`)  **Writes:** charts to `reports/figures/`

Every municipality loses people at two points on the way to the ballot box:

```
Adults who may vote  →  People who registered  →  People who voted
        E                        R                       V
            registration leak         turnout leak
```

- **Registration rate** `r = R / E` · **Turnout** `t = V / R` · **Real participation** `p = r × t = V / E`

This notebook is organised around **15 questions**. Each chart is saved as `qNN_...png`, named after the question it
answers. Each section asks one question, shows the evidence, and ends with
**Answer** (what the data shows) and, where needed, **Careful** (what it does *not* show). The last column says where
the chart can be used in the dashboard.

| # | Question | Dashboard |
|---|---|---|
| | **A. Turnout** | |
| Q1 | What was the turnout rate in 2011, 2016 and 2021? | Overview · Electoral participation |
| Q2 | How much did turnout fall in 2021, and did it fall everywhere? | Electoral participation |
| Q3 | How much do municipalities differ in turnout? | Electoral participation |
| Q4 | Which municipalities have the lowest turnout over the years? | Priority list · Municipality profile |
| | **B. Registration** | |
| Q5 | How many people were registered, 2011–2026? | Overview |
| Q6 | How has registration changed over the years, and where did it grow most? | Overview · Electoral participation |
| | **C. The funnel** | |
| Q7 | Where are more people lost — before registering, or after? | Overview |
| Q8 | Which problem does each municipality have, and where are they? | Map |
| Q9 | Are the registration leak and the turnout leak the same problem? | Recommendations |
| | **D. Who, and what goes with it** | |
| Q10 | Which age group has the lowest participation? | Recommendations · Municipality profile |
| Q11 | Which factors are linked to low turnout? | About · Recommendations |
| Q12 | Does low turnout cluster geographically? | Map |
| Q13 | Do metros differ from local municipalities? | Electoral participation |
| | **E. What to do** | |
| Q14 | Where should effort go before 4 November 2026? | Priority list · Recommendations |
| Q15 | Did past registration drives reach the places with the thinnest rolls? | Recommendations |

## 0. Setup

In [ ]:
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Thato20-M/democratic-funnel.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/democratic-funnel")
    url = REPO_URL
    try:
        from google.colab import userdata
        url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:
        print("No GITHUB_TOKEN in Colab Secrets - cloning will fail for the private repo")
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", url, str(REPO_DIR)], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "src" / "config.py").exists())

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Repo:", REPO_DIR)

In [ ]:
from src import config
from src.config import INTERIM, PROCESSED, FIGURES

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings; warnings.filterwarnings("ignore", category=FutureWarning)

config.make_dirs(); config.set_seed()
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 170)
print(config.describe())

### House style: one set of colours and words, used everywhere

The same colours mean the same thing in this notebook, on the slides and in the dashboard.

| Colour | Means |
|---|---|
| **Blue** | registration (the roll) |
| **Amber** | turnout (voting) |
| **Crimson** | both problems |
| **Green** | healthy |
| **Navy** | national figures |

The four group colours were checked for colour-blind readers as a set (worst pair ΔE 9.9 under colour-blind
simulation; target ≥ 8). Crimson and green are also far apart in lightness, so "both problems" and "healthy"
never look alike. Amber and light green are pale on white, so **every chart that uses them also has written
labels** — colour never carries meaning on its own.

`LABELS` turns column names into plain English, so nothing like `leak_turn` appears on a chart.

In [ ]:
# --- colours ---------------------------------------------------------------------------
BLUE, AMBER, CRIMSON, GREEN, NAVY = "#2a78d6", "#eda100", "#a3143a", "#7bc586", "#1b2f5b"
NAVY_LIGHT = "#b7c3dc"                      # context lines (e.g. individual provinces)
INK, INK2, GRID, SURFACE = "#0b0b0b", "#4a4a47", "#e4e3de", "#ffffff"

GROUPS = ["Low registration", "Low turnout", "Both low", "Healthy"]
GROUP_COLOURS = {"Low registration": BLUE, "Low turnout": AMBER, "Both low": CRIMSON, "Healthy": GREEN}
WHAT_TO_SEND = {"Low registration": "registration drive, ID and address help",
                "Low turnout": "voter education and mobilisation",
                "Both low": "both - first priority",
                "Healthy": "nothing urgent"}

# --- plain-English names for every column that appears on a chart or table -------------
LABELS = {
    "r": "Registration rate", "t": "Turnout", "p": "Real participation",
    "leak_reg": "Registration vs typical municipality", "leak_turn": "Turnout vs typical municipality",
    "registered": "Registered voters", "votes_cast": "Votes cast", "eligible_adults": "Adults who may vote",
    "eligible_youth": "Young adults who may vote (18-29)", "registered_youth": "Registered young adults (18-29)",
    "youth_r": "Youth registration rate (18-29)", "adult_r": "Registration rate, all adults",
    "higher_ed_rate": "Adults with higher education", "piped_water_rate": "Households with piped water",
    "electricity_rate": "Households with electricity", "refuse_removal_rate": "Households with refuse removal",
    "log_density": "Population density (log)", "pop_density": "People per km²", "area_km2": "Area (km²)",
    "spatial_lag_turnout": "Neighbours' turnout", "muni_name": "Municipality", "province": "Province",
    "muni_code": "Code", "group": "Problem group",
}

plt.rcParams.update({                         # large text: these charts go on slides
    "figure.dpi": 110, "savefig.dpi": 200, "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "axes.edgecolor": GRID, "axes.linewidth": 0.8,
    "axes.labelcolor": INK2, "axes.titlecolor": INK, "axes.titlesize": 14, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.titlepad": 12, "axes.labelsize": 11,
    "xtick.color": INK2, "ytick.color": INK2, "xtick.labelsize": 10.5, "ytick.labelsize": 10.5,
    "grid.color": GRID, "grid.linewidth": 0.7, "legend.frameon": False, "legend.fontsize": 10.5,
    "font.size": 11,
})


def tidy(ax, xgrid=False, ygrid=True):
    """Recessive axes: no box, gridlines behind the data only where they help reading."""
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(GRID)
    ax.set_axisbelow(True)
    ax.grid(ygrid, axis="y"); ax.grid(xgrid, axis="x")
    return ax


def pct(ax, axis="y"):
    (ax.yaxis if axis == "y" else ax.xaxis).set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))
    return ax


def save(fig, name):
    fig.savefig(FIGURES / f"{name}.png", bbox_inches="tight")
    print("saved", name)


def group_of(reg_gap, turn_gap):
    """Below the typical (median) municipality on registration and/or turnout."""
    return pd.Series(np.select([(reg_gap < 0) & (turn_gap < 0), reg_gap < 0, turn_gap < 0],
                               ["Both low", "Low registration", "Low turnout"], "Healthy"),
                     index=reg_gap.index)

## 1. Load and check the data

`master.csv` has one row per municipality per election: 213 municipalities × 4 elections
(2011, 2016, 2021, 2026) = 852 rows. 2026 has registration but no votes yet — that election has not happened.

In [ ]:
master = pd.read_csv(PROCESSED / "master.csv")
assert master.groupby("election")["muni_code"].nunique().to_dict() == {2011: 213, 2016: 213, 2021: 213, 2026: 213}
assert not master.duplicated(["election", "muni_code"]).any()

y21 = master[master["election"] == 2021].copy()
y26 = master[master["election"] == 2026].copy()
held = master[master["election"] < 2026].copy()          # elections that actually happened

missing = master.isna().groupby(master["election"]).sum().T
print("Missing values per column (only columns with any):")
print(missing[missing.sum(axis=1) > 0].to_string())

In [ ]:
# Use today's municipality names everywhere (some IEC files use older or placeholder names, e.g. "New Municipality")
current_name = y26.set_index("muni_code")["muni_name"]
current_name = current_name.where(~current_name.str.isupper(), current_name.str.title())
for df in (master, y21, y26, held):
    df["muni_name"] = df["muni_code"].map(current_name)

**Every missing value has a reason.** 2026 has no votes, turnout or turnout leak (not held yet). 2011 has no
"previous election" columns. `registered_youth` exists for 2026 only — the IEC publishes registration by
age for the current roll, not for past elections.

---
# A. Turnout

## Q1. What was the turnout rate in 2011, 2016 and 2021?

Turnout = votes cast ÷ registered voters, for the ward ballot, added up over all 213 municipalities.

In [ ]:
nat = held.groupby("election")[["registered", "votes_cast"]].sum()
nat.loc[2026, "registered"] = y26["registered"].sum()
nat["turnout"] = nat["votes_cast"] / nat["registered"]
print((nat.assign(registered=lambda d: (d.registered / 1e6).round(2),
                  votes_cast=lambda d: (d.votes_cast / 1e6).round(2),
                  turnout=lambda d: (d.turnout * 100).round(1))
       .rename(columns={"registered": "registered (m)", "votes_cast": "votes cast (m)", "turnout": "turnout %"})
       .to_string()))

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.6))
yrs = [2011, 2016, 2021]
ax.plot(yrs, nat.loc[yrs, "turnout"], "-o", color=AMBER, lw=2.8, ms=10, mec=SURFACE, mew=2)
for y in yrs:
    ax.annotate(f"{nat.loc[y, 'turnout']:.1%}", (y, nat.loc[y, "turnout"]), xytext=(0, 12),
                textcoords="offset points", ha="center", fontsize=12, color=INK, weight="bold")
ax.annotate("COVID election", (2021, nat.loc[2021, "turnout"]), xytext=(-14, -26),
            textcoords="offset points", ha="right", fontsize=10.5, color=INK2, style="italic")
ax.set_xticks(yrs); ax.set_xlim(2009.5, 2022.5); ax.set_ylim(0.35, 0.65); pct(ax); tidy(ax)
ax.set_title("Turnout was steady, then fell 12 points in 2021"); ax.set_ylabel("Votes cast ÷ registered voters")
save(fig, "q01_turnout"); plt.show()

**Answer.** National turnout was **57.4%** in 2011, **57.7%** in 2016 and **45.6%** in 2021 — steady for two
elections, then a fall of **12.1 points**.

**Careful.** Turnout counts only people who registered. Everyone missing from the roll is invisible in it (see Q7).

## Q2. How much did turnout fall in 2021, and did it fall everywhere?

In [ ]:
prov = (held.groupby(["election", "province"])[["votes_cast", "registered"]].sum()
            .pipe(lambda d: d["votes_cast"] / d["registered"]).unstack())

fig, ax = plt.subplots(figsize=(9, 5))
for p in prov.columns:
    ax.plot(prov.index, prov[p], color=NAVY_LIGHT, lw=1.4)
    ax.annotate(p, (2021, prov.loc[2021, p]), xytext=(6, 0), textcoords="offset points", va="center",
                fontsize=9.5, color=INK2)
ax.plot(nat.index[:3], nat["turnout"].iloc[:3], "-o", color=NAVY, lw=3, ms=8, label="National", zorder=5)
ax.set_xticks([2011, 2016, 2021]); ax.set_xlim(2010, 2024.5); pct(ax); tidy(ax)
ax.set_title("Turnout fell in every province in 2021"); ax.set_ylabel("Turnout")
ax.legend(loc="lower left")
save(fig, "q02_turnout_by_province"); plt.show()

change = (prov.loc[2021] - prov.loc[2016]).sort_values()
print("Change 2016 -> 2021 (points):"); print((change * 100).round(1).to_string())

**Answer.** **Every province fell.** The fall ranged from about **6 points in Limpopo** to about **15 points in the
Western Cape**; Gauteng, Mpumalanga and KwaZulu-Natal each fell 13–14 points.

**Careful.** A fall everywhere at once points to a national cause — the 2021 election was held under COVID-19
restrictions — but this data cannot prove the cause. It is why the model predicts the *change* in turnout and treats
the national swing separately.

## Q3. How much do municipalities differ in turnout?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))

ax = axes[0]
ax.hist(y21["r"].clip(upper=1.3), bins=28, color=NAVY, alpha=0.9, label="2021")
ax.hist(y26["r"].clip(upper=1.3), bins=28, color=BLUE, alpha=0.65, label="2026")
ax.axvline(1.0, color=CRIMSON, lw=1.4, ls="--")
ax.annotate("more registered\nthan adults counted", (1.0, ax.get_ylim()[1] * 0.8), xytext=(6, 0),
            textcoords="offset points", fontsize=10, color=CRIMSON)
pct(ax, "x"); tidy(ax); ax.set_title(LABELS["r"]); ax.set_xlabel("Registered ÷ adults who may vote")
ax.set_ylabel("Municipalities"); ax.legend(loc="upper left")

ax = axes[1]
for yr, col in [(2011, NAVY_LIGHT), (2016, "#e9c46a"), (2021, AMBER)]:
    ax.hist(held.loc[held["election"] == yr, "t"], bins=28, histtype="step", lw=2.4, color=col, label=str(yr))
pct(ax, "x"); tidy(ax); ax.set_title(LABELS["t"]); ax.set_xlabel("Votes cast ÷ registered")
ax.set_ylabel("Municipalities"); ax.legend(loc="upper left")

fig.suptitle("Municipalities differ a lot", x=0.06, ha="left", fontsize=15, weight="bold", y=1.03)
save(fig, "q03_turnout_spread"); plt.show()

print("Turnout range 2021:", f"{y21.t.min():.0%} - {y21.t.max():.0%}")
print("Registration rate above 100%: 2021", int((y21.r > 1).sum()), "| 2026", int((y26.r > 1).sum()))
print(y26.loc[y26.r > 1, "province"].value_counts().to_string())

**Answer.** In 2021 turnout ranged from **36% to 71%** between municipalities, and the whole distribution moved
lower than in 2011 and 2016. A few municipalities (4 in 2021, 15 for 2026 — over half in the Northern Cape) have more
registered voters than the Census counted adults.

**Careful.** Registration above 100% reflects the small Census sample in thinly populated places and a cumulative
voters' roll, not real over-registration. These municipalities are flagged, not deleted.

## Q4. Which municipalities have the lowest turnout over the years?

In [ ]:
tw = held.pivot(index="muni_code", columns="election", values="t")
tw["Average"] = tw[[2011, 2016, 2021]].mean(axis=1)
tw = tw.join(current_name.rename("Municipality")).join(y26.set_index("muni_code")["province"].rename("Province"))

print("Lowest turnout in each election:")
for y in (2011, 2016, 2021):
    low = tw.nsmallest(3, y)
    print(f"  {y}: " + "; ".join(f"{r.Municipality} ({r[y]:.1%})" for _, r in low.iterrows()))

bottom20 = [set(tw.nsmallest(20, y).index) for y in (2011, 2016, 2021)]
always = sorted(tw.loc[list(bottom20[0] & bottom20[1] & bottom20[2]), "Municipality"])
print("\nIn the bottom 20 in all three elections:", ", ".join(always))

tw["Fall 2016-2021"] = tw[2021] - tw[2016]
print("\nBiggest falls 2016 -> 2021 (points):")
print((tw.nsmallest(5, "Fall 2016-2021")[["Municipality", "Province", "Fall 2016-2021"]]
       .assign(**{"Fall 2016-2021": lambda d: (d["Fall 2016-2021"] * 100).round(1)}).to_string(index=False)))

low10 = tw.nsmallest(10, "Average").iloc[::-1]
fig, ax = plt.subplots(figsize=(10, 5.6))
for i, (yr, colour, dy) in enumerate([(2011, NAVY_LIGHT, -0.26), (2016, "#e9c46a", 0), (2021, AMBER, 0.26)]):
    ax.barh(np.arange(10) + dy, low10[yr], height=0.25, color=colour, label=str(yr))
ax.set_yticks(range(10)); ax.set_yticklabels([f"{m} ({p})" for m, p in zip(low10["Municipality"], low10["Province"])])
ax.axvline(nat.loc[2021, "turnout"], color=NAVY, lw=1.4, ls="--")
ax.annotate(f"national 2021: {nat.loc[2021, 'turnout']:.1%}", (nat.loc[2021, "turnout"], 9.55), xytext=(4, 0),
            textcoords="offset points", fontsize=10, color=NAVY)
pct(ax, "x"); tidy(ax, xgrid=True, ygrid=False); ax.set_xlim(0.3, 0.62)
ax.set_title("The 10 municipalities with the lowest average turnout, 2011–2021"); ax.set_xlabel("Turnout")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncols=3)
save(fig, "q04_lowest_turnout"); plt.show()

**Answer.** **Rustenburg (North West)** has the lowest turnout over the years: lowest in 2011 (45.0%) and 2021
(36.0%), and the lowest average across all three elections (43.7%). The lowest average turnout is concentrated in
**Limpopo** (7 of the 10) and **North West**. Six municipalities were in the bottom 20 in **every** election:
Rustenburg, King Sabata Dalindyebo, Thulamela, Fetakgomo Tubatse, Mogalakwena and Thabazimbi.

The biggest **falls** in 2021 were elsewhere — in KwaZulu-Natal and in metros: Mkhambathini (−23.6 points),
Msunduzi, Mbombela, eThekwini and Nelson Mandela Bay (about −18 to −19).

**Careful.** Low turnout is a percentage of *registered* voters. A municipality with low turnout may still have a
full roll — and vice versa (see Q8).

---
# B. Registration

## Q5. How many people were registered, 2011–2026?

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.8))
x = [str(y) for y in nat.index]
ax.bar(x, nat["registered"] / 1e6, color=[BLUE, BLUE, BLUE, "#8db8ec"], width=0.62)
E = y26["eligible_adults"].sum() / 1e6
ax.axhline(E, color=NAVY, lw=1.6, ls="--")
ax.annotate(f"Adults who may vote (Census 2022): {E:.1f} m", (-0.4, E), xytext=(0, 6),
            textcoords="offset points", fontsize=10.5, color=NAVY)
for i, v in enumerate(nat["registered"] / 1e6):
    ax.annotate(f"{v:.1f} m", (i, v), xytext=(0, 5), textcoords="offset points", ha="center", fontsize=12,
                weight="bold")
ax.set_ylim(0, E * 1.15); tidy(ax)
ax.set_title("A record 29.2 million registered for 2026"); ax.set_ylabel("Registered voters (millions)")
save(fig, "q05_registered"); plt.show()

reg_prov = master.pivot_table(index="province", columns="election", values="registered", aggfunc="sum")
print((reg_prov / 1e6).round(2).rename(columns=str).to_string())

**Answer.** **23.6 million** people were registered for 2011, **26.3 million** for 2016, **26.2 million** for 2021 and
a record **29.2 million** for 2026 (September 2026 snapshot). That is still about **10.8 million** short of the
40.1 million adults who may vote.

## Q6. How has registration changed over the years, and where did it grow most?

In [ ]:
steps = [(2011, 2016), (2016, 2021), (2021, 2026)]
change = pd.DataFrame({"period": [f"{a}→{b}" for a, b in steps],
                       "people": [nat.loc[b, "registered"] - nat.loc[a, "registered"] for a, b in steps],
                       "percent": [nat.loc[b, "registered"] / nat.loc[a, "registered"] - 1 for a, b in steps]})
print(change.assign(people=lambda d: (d.people / 1e6).round(2), percent=lambda d: (d.percent * 100).round(1))
      .rename(columns={"people": "change (m)", "percent": "change %"}).to_string(index=False))

growth = (reg_prov[2026] / reg_prov[2021] - 1).sort_values()
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw={"width_ratios": [1, 1.3]})
ax = axes[0]
ax.bar(change["period"], change["people"] / 1e6, color=[BLUE if v > 0 else CRIMSON for v in change["people"]],
       width=0.6)
for i, (v, p) in enumerate(zip(change["people"] / 1e6, change["percent"])):
    ax.annotate(f"{v:+.1f} m\n({p:+.1%})", (i, v), xytext=(0, 6 if v > 0 else -30), textcoords="offset points",
                ha="center", fontsize=11)
ax.axhline(0, color=INK2, lw=1); tidy(ax); ax.set_ylim(-0.8, 3.8)
ax.set_title("Change in registered voters"); ax.set_ylabel("Millions of people")
ax = axes[1]
ax.barh(growth.index, growth.values, color=BLUE, height=0.65)
for i, v in enumerate(growth.values):
    ax.annotate(f"+{v:.1%}", (v, i), xytext=(4, 0), textcoords="offset points", va="center", fontsize=10.5)
ax.set_xticks([0, 0.05, 0.10, 0.15, 0.20]); pct(ax, "x"); tidy(ax, xgrid=True, ygrid=False); ax.set_xlim(0, 0.21)
ax.set_title("Growth in registration, 2021 → 2026, by province")
fig.tight_layout(); save(fig, "q06_registration_change"); plt.show()

E26 = y26["eligible_adults"].sum()
print(f"Registration rate: 2021 {nat.loc[2021, 'registered'] / E26:.1%}  ->  2026 {nat.loc[2026, 'registered'] / E26:.1%}")

**Answer.** Registration grew by **2.7 million (+11.5%)** from 2011 to 2016, was **flat** from 2016 to 2021
(−0.1 million), then grew by **3.0 million (+11.6%)** for 2026. Growth for 2026 was strongest in **Limpopo (+17.5%)**
and weakest in the **Free State (+8.5%)** and **Gauteng (+8.9%)**. As a share of adults who may vote, registration rose
from about **65% (2021)** to **73% (2026)**.

**Careful.** We only have a population count for 2022, so registration *rates* are compared for 2021 and 2026 only.
Dividing 2011 or 2016 registration by the 2022 population would make those years look worse than they were. The 2026
rate is slightly overstated because the adult population has grown since 2022.

---
# C. The funnel

## Q7. Where are more people lost — before registering, or after?

In [ ]:
E21, R21, V21 = y21["eligible_adults"].sum(), y21["registered"].sum(), y21["votes_cast"].sum()
R26 = y26["registered"].sum()
vals21 = [100, 100 * R21 / E21, 100 * V21 / E21]

fig, ax = plt.subplots(figsize=(9, 3.8))
stages = ["May vote (adults)", "Registered", "Voted"]
ypos = np.arange(3)[::-1]
ax.barh(ypos + 0.19, vals21, height=0.34, color=NAVY, label="2021 (actual)")
ax.barh(ypos[:2] - 0.19, [100, 100 * R26 / E21], height=0.34, color=BLUE, label="2026 (roll so far)")
for y, v in zip(ypos + 0.19, vals21):
    ax.annotate(f"{v:.0f}", (v, y), xytext=(6, 0), textcoords="offset points", va="center", fontsize=12,
                color=INK, weight="bold")
for y, v in zip(ypos[:2] - 0.19, [100, 100 * R26 / E21]):
    ax.annotate(f"{v:.0f}", (v, y), xytext=(6, 0), textcoords="offset points", va="center", fontsize=12,
                color=BLUE, weight="bold")
ax.annotate("not yet held", (1, ypos[2] - 0.19), xytext=(4, 0), textcoords="offset points", va="center",
            fontsize=10.5, color=INK2, style="italic")
ax.set_yticks(ypos); ax.set_yticklabels(stages, fontsize=11.5)
ax.set_xlim(0, 115); ax.set_xlabel("per 100 adults who may vote")
tidy(ax, xgrid=True, ygrid=False)
ax.set_title("Out of every 100 adults, 65 were registered and 30 voted (2021)")
ax.legend(loc="lower right")
save(fig, "q07_funnel"); plt.show()

print(f"2021: {100*R21/E21:.0f} registered, {100*V21/E21:.0f} voted per 100 adults")
print(f"2026: {100*R26/E21:.0f} registered per 100 adults (so far)")

**Answer.** In 2021, of every 100 adults who may vote, **65** were registered and **30** voted. About **35** were lost
before registering and another **35** after registering — the two losses are about the same size. By September 2026,
**73** of every 100 are on the roll.

**Careful.** The published turnout figure only shows the second loss.

## Q8. Which problem does each municipality have, and where are they?

Each municipality is compared with the **typical (median) municipality** of that election:
registration vs typical = ln(r / median r); turnout vs typical = ln(t / median t). Below 0 means worse than typical.
The logarithms make the two add up exactly to the municipality's total participation gap. The signs place every
municipality in one of four groups.

In [ ]:
y21["group"] = group_of(y21["leak_reg"], y21["leak_turn"])
counts21 = y21["group"].value_counts().reindex(GROUPS)
print(counts21.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7.4))
for name in GROUPS:
    s = y21[y21["group"] == name]
    ax.scatter(s["leak_reg"].clip(-0.6, 0.6), s["leak_turn"], s=48, color=GROUP_COLOURS[name],
               edgecolor=SURFACE, linewidth=0.9, label=f"{name} ({len(s)})")
ax.axhline(0, color=INK2, lw=1); ax.axvline(0, color=INK2, lw=1)
lim = 0.62
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
corners = [(-lim * .96, lim * .9, "LOW REGISTRATION", "few registered - but those\nwho did, voted", "left"),
           (lim * .96, lim * .9, "HEALTHY", "above typical on both", "right"),
           (-lim * .96, -lim * .62, "BOTH LOW", "few registered, and few\nof those voted", "left"),
           (lim * .96, -lim * .62, "LOW TURNOUT", "the roll is full - but\nvoters stayed home", "right")]
for x, y, head, sub, ha in corners:
    ax.text(x, y, head, fontsize=11, color=INK, weight="bold", ha=ha, va="bottom")
    ax.text(x, y, sub, fontsize=9.5, color=INK2, ha=ha, va="top", linespacing=1.35)
ax.set_xlabel("Registration vs typical municipality   ←  worse    better  →")
ax.set_ylabel("Turnout vs typical municipality   ←  worse    better  →")
tidy(ax, xgrid=True)
ax.set_title("2021: every municipality falls into one of four groups")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncols=2)
save(fig, "q08_problem_groups"); plt.show()

In [ ]:
bounds_path = INTERIM / "boundaries.gpkg"
if bounds_path.exists():
    import geopandas as gpd
    gmap = gpd.read_file(bounds_path).merge(y21[["muni_code", "group"]], on="muni_code", how="left")
    fig, ax = plt.subplots(figsize=(9, 9))
    for name in GROUPS:
        sub = gmap[gmap["group"] == name]
        if len(sub):
            sub.plot(ax=ax, color=GROUP_COLOURS[name], edgecolor=SURFACE, linewidth=0.4)
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color=GROUP_COLOURS[g], label=f"{g} ({counts21[g]})") for g in GROUPS],
              loc="lower left", fontsize=11)
    ax.set_axis_off(); ax.set_title("2021: which problem, where")
    save(fig, "q08_problem_map"); plt.show()
else:
    print(f"{bounds_path} not found - run 05_geography first. Skipping the map.")

**Answer.** The 213 municipalities split into **68 healthy, 39 low registration, 39 low turnout and 67 low on both**
(2021). The groups cluster by region rather than scattering at random.

| Group | What to send |
|---|---|
| Low registration | registration drive, ID and address help |
| Low turnout | voter education and mobilisation |
| Both low | both — first priority |
| Healthy | nothing urgent |

## Q9. Are the registration leak and the turnout leak the same problem?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7.2))
for ax, src, col, colour, title, xlabel in [
        (axes[0], y26, "r", BLUE, "Thinnest rolls, 2026", "Registered ÷ adults who may vote"),
        (axes[1], y21, "t", AMBER, "Lowest turnout, 2021", "Votes cast ÷ registered")]:
    w = src.nsmallest(20, col).sort_values(col, ascending=False)
    ax.barh(range(20), w[col], color=colour, height=0.72)
    ax.set_yticks(range(20)); ax.set_yticklabels([n[:26] for n in w["muni_name"]], fontsize=10.5)
    med = src[col].median()
    ax.axvline(med, color=NAVY, lw=1.4, ls="--")
    ax.annotate(f"typical: {med:.0%}", (med, 19.5), xytext=(4, 0), textcoords="offset points",
                fontsize=10, color=NAVY)
    for i, v in enumerate(w[col]):
        ax.annotate(f"{v:.0%}", (v, i), xytext=(4, 0), textcoords="offset points", va="center", fontsize=9.5)
    pct(ax, "x"); tidy(ax, xgrid=True, ygrid=False); ax.set_title(title); ax.set_xlabel(xlabel)
fig.tight_layout(); save(fig, "q09_two_leaks"); plt.show()

overlap = set(y26.nsmallest(20, "r").muni_code) & set(y21.nsmallest(20, "t").muni_code)
print("Municipalities on both lists:", len(overlap))

In [ ]:
print("Correlation between the two leaks (2021):", round(y21["leak_reg"].corr(y21["leak_turn"]), 2))

**Answer.** **No.** The 20 thinnest rolls and the 20 lowest turnouts share only **2** municipalities, and the two
measures are only weakly related (correlation **0.30**). Two municipalities with the same turnout can have opposite
problems — which is why the published turnout figure is not enough to plan a response.

---
# D. Who, and what goes with low participation

## Q10. Which age group has the lowest participation?

The IEC publishes 2026 registration by age. We compare **registered 18–29** with **adults aged 18–29 who may vote**
(Census 2022), and with everyone aged 30 and older.

In [ ]:
y26["youth_r"] = y26["registered_youth"] / y26["eligible_youth"]
y26["adult_r"] = y26["registered"] / y26["eligible_adults"]
EY, RY = y26["eligible_youth"].sum(), y26["registered_youth"].sum()
EA, RA = y26["eligible_adults"].sum(), y26["registered"].sum()
share_adults, share_missing = EY / EA, (EY - RY) / (EA - RA)

print(f"Aged 18-29: {share_adults:.0%} of adults who may vote")
print(f"Aged 18-29: {share_missing:.0%} of everyone missing from the roll ({(EY - RY) / 1e6:.1f} m people)")
print(f"Registration rate 18-29: {RY / EY:.0%}   |   30 and older: {(RA - RY) / (EA - EY):.0%}")
print(f"Municipalities where youth register less than adults overall: {(y26.youth_r < y26.adult_r).sum()} of 213")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [1.15, 1]})

ax = axes[0]
for i, (lab, share, n_y, n_o) in enumerate([("All adults who may vote", share_adults, EY, EA - EY),
                                            ("Missing from the roll", share_missing, EY - RY,
                                             (EA - RA) - (EY - RY))]):
    y = 1 - i
    ax.barh(y, share, height=0.5, color=BLUE)
    ax.barh(y, 1 - share, left=share + 0.006, height=0.5, color=NAVY_LIGHT)
    ax.annotate(f"18-29\n{share:.0%} ({n_y / 1e6:.1f} m)", (share / 2, y), ha="center", va="center",
                fontsize=11, color="white", weight="bold")
    ax.annotate(f"30+\n{1 - share:.0%} ({n_o / 1e6:.1f} m)", (share + (1 - share) / 2, y), ha="center",
                va="center", fontsize=11, color=INK)
ax.set_yticks([1, 0]); ax.set_yticklabels(["All adults who may vote", "Missing from the roll"], fontsize=11.5)
ax.set_xlim(0, 1.02); ax.set_xticks([]); ax.set_ylim(-0.6, 1.6)
for sp in ax.spines.values():
    sp.set_visible(False)
ax.set_title(f"Young adults: {share_adults:.0%} of adults, {share_missing:.0%} of those missing")

ax = axes[1]
ax.scatter(y26["adult_r"].clip(upper=1.2), y26["youth_r"].clip(upper=1.2), s=36, color=BLUE,
           edgecolor=SURFACE, linewidth=0.8, alpha=0.9)
ax.plot([0.3, 1.2], [0.3, 1.2], color=NAVY, lw=1.4, ls="--")
ax.annotate("youth register as often\nas all adults", (1.0, 1.0), xytext=(-8, 10), textcoords="offset points",
            ha="right", fontsize=10, color=NAVY)
pct(ax); pct(ax, "x"); tidy(ax, xgrid=True)
ax.set_xlabel(LABELS["adult_r"]); ax.set_ylabel(LABELS["youth_r"])
ax.set_title("212 of 213 municipalities sit below the line")
fig.tight_layout(); save(fig, "q10_youth"); plt.show()

**Answer.** **Young adults aged 18–29.** Only **46%** of them are registered, against **84%** of adults aged 30 and
older. They are **30%** of adults who may vote but **60%** of everyone missing from the roll — about **6.5 million**
people. In **212 of 213** municipalities young adults register less than adults overall.

**Careful.** This is about **registration only**. The IEC does not publish *turnout* by age for municipalities, so we
cannot say which age group votes least once registered — and we do not claim it.

## Q11. Which factors are linked to low turnout?

We look at how turnout, and the fall in turnout in 2021, move together with municipal characteristics.

**A link is not proof of a cause.** A pattern across municipalities cannot show why individuals do or do not vote
(the ecological fallacy). The honest version of "what contributes to low turnout" is "what is **associated** with it".

In [ ]:
y21["log_density"] = np.log(y21["pop_density"])
feats = ["leak_reg", "leak_turn", "higher_ed_rate", "piped_water_rate", "electricity_rate",
         "refuse_removal_rate", "log_density", "spatial_lag_turnout"]
corr = y21[feats].corr()
names = [LABELS[f] for f in feats]

fig, ax = plt.subplots(figsize=(9.5, 8))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(feats))); ax.set_xticklabels(names, rotation=40, ha="right", fontsize=10.5)
ax.set_yticks(range(len(feats))); ax.set_yticklabels(names, fontsize=10.5)
for i in range(len(feats)):
    for j in range(len(feats)):
        v = corr.iloc[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=9.5, color="white" if abs(v) > .55 else INK)
ax.set_title("How municipal characteristics move together, 2021")
for s in ax.spines.values():
    s.set_visible(False)
fig.colorbar(im, ax=ax, shrink=0.7, label="correlation")
save(fig, "q11_correlations"); plt.show()

print(corr[["leak_reg", "leak_turn"]].rename(index=LABELS, columns=LABELS).round(2).to_string())

In [ ]:
y21["Density quarter"] = pd.qcut(y21["pop_density"], 4, labels=["Least dense", "2nd", "3rd", "Most dense"])
dq = y21.groupby("Density quarter", observed=True).agg(turnout=("t", "median"), fall=("dt", "median"))

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
axes[0].bar(dq.index.astype(str), dq["turnout"], color=AMBER, width=0.6)
for i, v in enumerate(dq["turnout"]):
    axes[0].annotate(f"{v:.0%}", (i, v), xytext=(0, 5), textcoords="offset points", ha="center", fontsize=12)
pct(axes[0]); tidy(axes[0]); axes[0].set_ylim(0, 0.62)
axes[0].set_title("Typical turnout, 2021"); axes[0].set_xlabel("Municipalities grouped by population density")
axes[1].bar(dq.index.astype(str), dq["fall"] * 100, color=CRIMSON, width=0.6)
for i, v in enumerate(dq["fall"] * 100):
    axes[1].annotate(f"{v:.1f}", (i, v), xytext=(0, -18), textcoords="offset points", ha="center", fontsize=12)
tidy(axes[1]); axes[1].set_ylim(-14.5, 0); axes[1].set_title("Typical fall in turnout, 2016 → 2021 (points)")
axes[1].set_xlabel("Municipalities grouped by population density")
fig.suptitle("The densest municipalities vote least, and fell furthest in 2021", x=0.06, ha="left", fontsize=15,
             weight="bold", y=1.04)
save(fig, "q11_density"); plt.show()

print("Correlation with turnout 2021:")
print(y21[["log_density", "spatial_lag_turnout", "r", "higher_ed_rate", "piped_water_rate",
           "refuse_removal_rate"]].corrwith(y21["t"]).rename(index=LABELS | {"r": "Registration rate"}).round(2).to_string())

**Answer.** Low turnout is most strongly linked with:

1. **Population density.** Denser municipalities had lower turnout in 2021 (correlation **−0.51**): the least dense
   quarter had a typical turnout of **53%**, the densest **45%** — and the densest fell furthest in 2021 (**−12 points**
   against about −8).
2. **Neighbours' turnout** (**0.64**): low-turnout municipalities sit next to other low-turnout municipalities (Q12).
3. **Registration** (**0.30**): places with fuller rolls tend to have slightly higher turnout.

Household services (water, electricity, refuse removal) and higher education show **little link** with the level of
turnout (between −0.02 and 0.13). They are more strongly linked to the *fall* in 2021, but they also move closely
together, so they partly measure the same thing.

**Careful.** These are associations, not causes. Explanations such as transport, work, trust or information are
possible reasons that need other evidence; this data cannot test them. The model in `08_model` uses these factors
to predict turnout, not to explain it.

## Q12. Does low turnout cluster geographically?

In [ ]:
sp = y21.dropna(subset=["spatial_lag_turnout", "t"])
rho = sp["t"].corr(sp["spatial_lag_turnout"])
rho_prev = y21["t"].corr(y21["spatial_lag_prev"]) if "spatial_lag_prev" in y21 else np.nan

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(sp["spatial_lag_turnout"], sp["t"], s=36, color=AMBER, edgecolor=SURFACE, linewidth=0.8)
b, a = np.polyfit(sp["spatial_lag_turnout"], sp["t"], 1)
xs = np.linspace(sp["spatial_lag_turnout"].min(), sp["spatial_lag_turnout"].max(), 50)
ax.plot(xs, a + b * xs, color=NAVY, lw=2.2)
ax.annotate(f"correlation = {rho:.2f}", (0.04, 0.93), xycoords="axes fraction", fontsize=12, color=NAVY,
            weight="bold")
pct(ax); pct(ax, "x"); tidy(ax, xgrid=True)
ax.set_xlabel("Neighbours' average turnout, 2021"); ax.set_ylabel("Own turnout, 2021")
ax.set_title("Turnout clusters geographically")
save(fig, "q12_clustering"); plt.show()
print(f"Same election: {rho:.2f}   |   neighbours' turnout at the previous election: {rho_prev:.2f}")

**Answer.** **Yes.** A municipality's turnout is strongly related to its neighbours' turnout in the same election
(correlation **0.64**), and still clearly related to their turnout at the previous election (**0.54**). Low turnout is
regional — e.g. across parts of Limpopo, North West and Mpumalanga.

**Careful.** Clustering suggests regional causes but does not identify them. The model uses the *previous-election*
version, because when forecasting 2026 nobody's 2026 turnout is known yet.

## Q13. Do metros differ from local municipalities?

Eight metros hold about 40% of all registered voters.

In [ ]:
lbl = {"A": "Metros (8)", "B": "Local (205)"}
fig, axes = plt.subplots(1, 3, figsize=(13, 4.4))
for ax, (src, col, title, colour) in zip(axes, [(y26, "r", "Registration rate, 2026", BLUE),
                                                (y21, "t", "Turnout, 2021", AMBER),
                                                (y26, "youth_r", "Youth registration, 2026", BLUE)]):
    data = [src.loc[src["muni_category"] == c, col].dropna().clip(upper=1.2) for c in ["A", "B"]]
    bp = ax.boxplot(data, patch_artist=True, widths=0.5, medianprops=dict(color=INK, lw=2.2),
                    flierprops=dict(marker="o", ms=4, mfc=NAVY_LIGHT, mec=SURFACE))
    for patch, c in zip(bp["boxes"], [colour, NAVY_LIGHT]):
        patch.set_facecolor(c); patch.set_edgecolor(SURFACE)
    ax.set_xticks([1, 2]); ax.set_xticklabels([lbl["A"], lbl["B"]]); pct(ax); tidy(ax); ax.set_title(title)
fig.suptitle("Metros register less, vote less, and their young adults register least", x=0.06, ha="left",
             fontsize=15, weight="bold", y=1.04)
save(fig, "q13_metros"); plt.show()

print(pd.DataFrame({"Registration 2026": y26.groupby("muni_category")["r"].median(),
                    "Youth registration 2026": y26.groupby("muni_category")["youth_r"].median(),
                    "Turnout 2021": y21.groupby("muni_category")["t"].median()})
      .rename(index=lbl).mul(100).round(1).to_string())

**Answer.** **Yes.** The typical metro is below the typical local municipality on all three measures: registration
**69% vs 79%**, turnout **44% vs 49%**, and youth registration **41% vs 55%**. Metros also had some of the largest falls
in 2021.

**Why it matters.** The largest numbers of missing people — especially young adults — are in the metros, which points
to urban youth registration campaigns alongside rural registration drives.

---
# E. What to do

## Q14. Where should effort go before 4 November 2026?

2026 has no turnout yet, so we combine **what we know now** (the 2026 roll) with **what happened last time** (2021
turnout), and turn each shortfall into **people**: how many more registrations, and how many more voters, would bring
the municipality up to the typical level. *(The dashboard uses the model's predicted 2026 turnout instead.)*

In [ ]:
prio = y26[["muni_code", "muni_name", "province", "leak_reg", "r", "registered", "eligible_adults"]].merge(
    y21[["muni_code", "leak_turn", "t"]], on="muni_code")
prio["group"] = group_of(prio["leak_reg"], prio["leak_turn"])
prio["shortfall"] = -(prio["leak_reg"] + prio["leak_turn"])            # bigger = further below typical
r_typ, t_typ = prio["r"].median(), prio["t"].median()
prio["registrations_needed"] = (r_typ * prio["eligible_adults"] - prio["registered"]).clip(lower=0).round(-1)
prio["extra_voters_needed"] = ((t_typ - prio["t"]) * prio["registered"]).clip(lower=0).round(-1)
prio["what_to_send"] = prio["group"].map(WHAT_TO_SEND)

top = prio.nlargest(15, "shortfall")
print(f"Below typical on both measures: {(prio.group == 'Both low').sum()} municipalities\n")
print(top[["muni_name", "province", "r", "t", "registrations_needed", "extra_voters_needed", "what_to_send"]]
      .rename(columns=LABELS | {"r": "Registration 2026", "t": "Turnout 2021",
                                "registrations_needed": "Registrations needed",
                                "extra_voters_needed": "Extra voters needed", "what_to_send": "What to send"})
      .assign(**{"Registration 2026": lambda d: (d["Registration 2026"] * 100).round(1),
                 "Turnout 2021": lambda d: (d["Turnout 2021"] * 100).round(1)})
      .to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8.4, 6.8))
ax.scatter(prio["leak_reg"].clip(-0.6, 0.6), prio["leak_turn"], s=34, color=NAVY_LIGHT, edgecolor=SURFACE,
           linewidth=0.8, label="all municipalities")
ax.scatter(top["leak_reg"].clip(-0.6, 0.6), top["leak_turn"], s=70, color=CRIMSON, edgecolor=SURFACE,
           linewidth=1, label="15 furthest below typical", zorder=4)
for _, row in top.head(8).iterrows():
    ax.annotate(row["muni_name"][:24], (max(row["leak_reg"], -0.6), row["leak_turn"]), xytext=(6, 4),
                textcoords="offset points", fontsize=10, color=INK)
ax.axhline(0, color=INK2, lw=1); ax.axvline(0, color=INK2, lw=1)
ax.set_xlabel("Registration vs typical, 2026 roll   ←  worse")
ax.set_ylabel("Turnout vs typical, 2021   ←  worse")
tidy(ax, xgrid=True); ax.set_title("Priority for the weeks before 4 November"); ax.legend(loc="upper left")
save(fig, "q14_priority"); plt.show()

**Answer.** **67** municipalities are below typical on both the 2026 roll and 2021 turnout. Of the 15 furthest below,
**5 are in Mpumalanga** (Mkhondo, Msukaligwa, Thembisile Hani, Dr Pixley Ka Isaka Seme, Nkomazi) and **4 in Gauteng**
(Emfuleni, Ekurhuleni, Rand West City, Merafong City). Ekurhuleni alone is about 450 000 registrations short of typical.

## Q15. Did past registration drives reach the places with the thinnest rolls?

From the IEC Atlas: new registrations per municipality on the 2016 registration weekends. (Skipped if the Atlas file
from `05_geography` is not available.)

In [ ]:
act_path = INTERIM / "atlas_registration_activity_2016.csv"
if act_path.exists():
    act16 = pd.read_csv(act_path)
    a16 = master[master["election"] == 2016].merge(act16, on="muni_code", how="left")
    a16["new_reg_rate"] = a16["NEWREG3"] / a16["registered"]
    fig, ax = plt.subplots(figsize=(7.6, 6))
    ax.scatter(a16["new_reg_rate"], a16["r"].clip(upper=1.2), s=36, color=BLUE, edgecolor=SURFACE, linewidth=0.8)
    pct(ax); pct(ax, "x"); tidy(ax, xgrid=True)
    ax.set_xlabel("New registrations as a share of the roll, 2016"); ax.set_ylabel("Registration rate, 2016")
    ax.set_title("Did registration drives reach the thinnest rolls?")
    save(fig, "q15_registration_effort"); plt.show()
    print("correlation:", round(a16["new_reg_rate"].corr(a16["r"]), 3))
else:
    print(f"{act_path} not found - run 05_geography first. Skipping.")

**How to read it.** A *negative* correlation would mean effort went where the roll was thinnest — what you would
want. A flat or positive one would mean drives went where registration was already good.

**Careful.** The Atlas publishes this for 2011 and 2016 only, so it cannot be used for 2026.

---
## Summary: the answers

| # | Question | Answer |
|---|---|---|
| Q1 | Turnout 2011–2021 | **57.4% → 57.7% → 45.6%**: steady, then −12.1 points in 2021 |
| Q2 | Did it fall everywhere? | **Yes**, every province — from −6 (Limpopo) to −15 points (Western Cape) |
| Q3 | How much do municipalities differ? | 2021 turnout ranged from **36% to 71%** |
| Q4 | Lowest turnout over the years | **Rustenburg** (average 43.7%); mostly **Limpopo** and **North West**; 6 municipalities in the bottom 20 every time |
| Q5 | Registered 2011–2026 | **23.6 m → 26.3 m → 26.2 m → 29.2 m** (record); 10.8 m adults still not registered |
| Q6 | How registration changed | +2.7 m, flat, then **+3.0 m** for 2026; fastest growth in **Limpopo (+17.5%)**; rate 65% → 73% |
| Q7 | Where are people lost? | About equally: **35 of 100** before registering, **35** after (2021) |
| Q8 | Which problem, where? | **68** healthy · **39** low registration · **39** low turnout · **67** both; regional clusters |
| Q9 | Same problem? | **No**: 2 of 20 overlap; correlation 0.30 |
| Q10 | Lowest-participating age group | **18–29**: 46% registered vs 84% of 30+; 60% of everyone missing (registration only) |
| Q11 | Factors linked to low turnout | **Density** (−0.51), **neighbours' turnout** (0.64), registration (0.30); services and education barely linked |
| Q12 | Does it cluster? | **Yes** (0.64 with neighbours) |
| Q13 | Metros vs local | Metros lower on registration, turnout and especially youth registration |
| Q14 | Where to act first | 67 below typical on both; top 15 mostly **Mpumalanga** and **Gauteng** |

### What we are careful not to claim

- **A link is not proof of a cause.** Patterns across municipalities cannot show how individuals behave.
- **Youth figures are about registration only.** The IEC does not publish turnout by age at municipal level.
- **Registration rates use the 2022 Census** for every year, so we compare rates only for 2021 and 2026.
- **Registration above 100%** in a few small municipalities reflects the Census sample and a cumulative roll.

In [ ]:
print("Figures written to", FIGURES)
for f in sorted(FIGURES.glob("*.png")):
    print(" ", f.name)